# Clase 02 — Reproducibilidad

**Curso:** MLOps y Monitorización con Python · **Bloque 1:** El problema y la base
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/02_reproducibilidad.md`

## Qué vas a ver en esta clase

- Por qué **"en mi máquina funciona"** es un síntoma de un problema de reproducibilidad, no una excusa.
- **Dependencias bloqueadas**: generar un `requirements.txt` y un `pyproject.toml` reales, con las versiones exactas instaladas en este entorno.
- **Todas** las fuentes de aleatoriedad de un proyecto pequeño (NumPy, división train/test, el propio modelo) y cuáles se olvidan más a menudo.
- Una **estructura de carpetas** con responsabilidades claras (`src/`, `config/`, `tests/`), inspirada en Cookiecutter Data Science.
- **Separar la configuración del código** con un `config.yaml`.
- La refactorización guiada, con código ejecutado de verdad, de `datos/notebook_sucio.ipynb` a `proyecto_churn/`: la demostración final compara dos ejecuciones de la misma *pipeline* y prueba que una es reproducible y la otra no.

Este notebook crea una carpeta de trabajo efímera, `proyecto_churn/`, junto a este mismo notebook (se recrea vacía cada vez que ejecutas la Celda 0, así que siempre partes de cero).

## Requisitos

- Librerías: `numpy`, `pandas`, `scikit-learn` y `PyYAML`. La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso, con `notebook_sucio.ipynb` dentro (la abrimos solo como referencia, igual que en la clase 1).

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez.

---
## Instalación

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("PyYAML", "yaml")
print("Librerías listas.")

---
## Celda 0 — Setup

Cargamos `notebook_sucio.ipynb` como referencia (sin ejecutarlo, igual que en la clase 1) y preparamos `proyecto_churn/`, la carpeta de trabajo efímera que iremos llenando durante la clase.

In [ ]:
import os
import re
import sys
import json
import shutil
import subprocess
import importlib.util
from importlib.metadata import version

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

RUTA_SUCIO = cargar_datos("notebook_sucio.ipynb")
sucio = json.load(open(RUTA_SUCIO, encoding="utf-8"))
print("Notebook sucio cargado (solo como referencia, no se ejecuta):", RUTA_SUCIO)

# Carpeta de trabajo efímera: se recrea vacía en cada ejecución
WORK = "proyecto_churn"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)
print("Carpeta de trabajo lista:", os.path.abspath(WORK))

---
## 1. "En mi máquina funciona" no es una excusa

Reproducimos la celda 4 del notebook sucio (sin la parte del archivo "extra") en una función, y la llamamos dos veces seguidas, en el mismo notebook, sin fijar ninguna semilla.

In [ ]:
def generar_datos_sin_semilla(n=400):
    """Misma lógica que la celda 4 de notebook_sucio.ipynb, sin ninguna semilla."""
    datos = pd.DataFrame({
        "antiguedad_meses": np.random.randint(1, 72, n),
        "uso_mensual_horas": np.random.gamma(2, 5, n),
        "tickets_soporte": np.random.poisson(1.5, n),
        "factura_mensual": np.random.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (np.random.rand(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


datos_1 = generar_datos_sin_semilla(400)
datos_2 = generar_datos_sin_semilla(400)

print("filas generadas en cada llamada:", len(datos_1), len(datos_2))
print("¿las dos llamadas dan el mismo DataFrame?", datos_1.equals(datos_2))
assert not datos_1.equals(datos_2)

**¿Qué observas?** Las dos llamadas usan exactamente la misma función y el mismo argumento (`n=400`), y aun así el `DataFrame` no es idéntico (`datos_1.equals(datos_2)` da `False`). Ninguna máquina tiene la culpa: el problema es que `generar_datos_sin_semilla` nunca fija con qué estado empieza `np.random`. Es exactamente el mismo defecto que la celda 4 del notebook sucio.

---
## 2. Dependencias bloqueadas: `requirements.txt` y `pyproject.toml`

Generamos ambos archivos con las versiones **realmente instaladas** en este entorno, usando `importlib.metadata.version`, no escribiéndolas a mano.

In [ ]:
PAQUETES = [("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "scikit-learn"), ("PyYAML", "PyYAML")]
versiones = {nombre_pip: version(nombre_pip) for nombre_pip, _ in PAQUETES}

lineas_requirements = [f"{paquete}=={ver}" for paquete, ver in versiones.items()]
with open(os.path.join(WORK, "requirements.txt"), "w") as f:
    f.write("\n".join(lineas_requirements) + "\n")

print("requirements.txt generado:")
print(open(os.path.join(WORK, "requirements.txt")).read())

In [ ]:
dependencias_toml = ",\n    ".join(f'"{paquete}=={ver}"' for paquete, ver in versiones.items())
pyproject = f"""[project]
name = "proyecto_churn"
version = "0.1.0"
dependencies = [
    {dependencias_toml}
]
"""
with open(os.path.join(WORK, "pyproject.toml"), "w") as f:
    f.write(pyproject)

print("pyproject.toml generado (formato alternativo, misma información):")
print(open(os.path.join(WORK, "pyproject.toml")).read())

**¿Qué observas?** Ambos archivos fijan las mismas 4 versiones exactas, en dos formatos distintos. Ninguno de los dos dice nada sobre la versión de Python, el sistema operativo o las librerías del sistema: eso es lo que resuelve Docker, en la clase 8 (aquí solo lo mencionamos).

---
## 3. Fijar semillas en todas las fuentes de aleatoriedad

Reescribimos `generar_datos_sin_semilla` con un generador propio (`np.random.default_rng(seed)`) que recibe la semilla como parámetro, en vez de tocar el estado global.

In [ ]:
def generar_datos(n, seed):
    """Misma lógica, con generador propio y semilla explícita."""
    rng = np.random.default_rng(seed)
    datos = pd.DataFrame({
        "antiguedad_meses": rng.integers(1, 72, n),
        "uso_mensual_horas": rng.gamma(2, 5, n),
        "tickets_soporte": rng.poisson(1.5, n),
        "factura_mensual": rng.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (rng.random(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


misma_semilla_1 = generar_datos(400, seed=42)
misma_semilla_2 = generar_datos(400, seed=42)
otra_semilla = generar_datos(400, seed=7)

print("misma semilla → mismo DataFrame:", misma_semilla_1.equals(misma_semilla_2))
print("semilla distinta → DataFrame distinto:", not misma_semilla_1.equals(otra_semilla))
assert misma_semilla_1.equals(misma_semilla_2)
assert not misma_semilla_1.equals(otra_semilla)

**¿Qué observas?** Con `generar_datos(400, seed=42)`, dos llamadas seguidas dan el mismo `DataFrame` byte a byte; con `seed=7`, el resultado cambia, como es de esperar. Fijar la semilla no elimina la aleatoriedad: la convierte en **pseudoaleatoriedad controlada** (sección 3 del capítulo). Falta todavía fijar la división train/test y el modelo — lo hacemos en la sección 6.

---
## 4. Estructura de carpetas: `src/`, `config/`, `tests/`

Creamos la estructura mínima dentro de `proyecto_churn/`, inspirada en Cookiecutter Data Science (capítulo, sección 4), sin copiar su plantilla completa.

In [ ]:
for carpeta in ("src", "config", "tests"):
    os.makedirs(os.path.join(WORK, carpeta), exist_ok=True)

print(f"Estructura de {WORK}/:")
for raiz, subcarpetas, archivos in os.walk(WORK):
    subcarpetas.sort()
    nivel = raiz.replace(WORK, "").count(os.sep)
    print("  " * nivel + os.path.basename(raiz or WORK) + "/")
    for archivo in sorted(archivos):
        print("  " * (nivel + 1) + archivo)

**¿Qué observas?** `proyecto_churn/` ya tiene tres carpetas con responsabilidad clara y ningún archivo suelto en la raíz salvo `requirements.txt` y `pyproject.toml`. Compáralo con el "Proyecto A" del capítulo 1 (sección 3.1): un único `.ipynb` donde todo — datos, entrenamiento, evaluación — vive en el mismo archivo.

---
## 5. Separar la configuración del código

Ningún hiperparámetro ni semilla queda escrito dentro de una función: todos viven en `config/config.yaml`.

In [ ]:
config = {
    "n_filas": 400,
    "test_size": 0.25,
    "hiperparametros": {"n_estimators": 100, "max_depth": 5},
    "semillas": {"datos": 42, "split": 42, "modelo": 42},
}

with open(os.path.join(WORK, "config", "config.yaml"), "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("config.yaml escrito. Contenido:")
print(open(os.path.join(WORK, "config", "config.yaml")).read())

config_releido = yaml.safe_load(open(os.path.join(WORK, "config", "config.yaml")))
print("¿lo que se leyó es igual a lo que se escribió?", config_releido == config)
assert config_releido == config

**¿Qué observas?** El diccionario `config` que se escribió y el que se releyó desde disco son exactamente iguales: `config.yaml` es una representación fiel, en texto plano, de los parámetros del proyecto. Cambiar `"n_estimators": 100` a `200` sería editar una línea de este archivo, sin tocar `src/` (que todavía no existe: lo creamos en la sección siguiente).

---
## 6. La lógica en `src/pipeline.py`, y la prueba final de reproducibilidad

Escribimos el código de la *pipeline* en un archivo real, `src/pipeline.py` (no en una celda suelta), con tres funciones: `generar_datos`, `dividir` y `entrenar`, todas con parámetros explícitos y ninguna variable global. Después lo importamos y lo ejecutamos dos veces.

In [ ]:
codigo_pipeline = '''
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


def generar_datos(n, seed):
    rng = np.random.default_rng(seed)
    datos = pd.DataFrame({
        "antiguedad_meses": rng.integers(1, 72, n),
        "uso_mensual_horas": rng.gamma(2, 5, n),
        "tickets_soporte": rng.poisson(1.5, n),
        "factura_mensual": rng.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (rng.random(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


def dividir(datos, config):
    columnas = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
    X, y = datos[columnas], datos["abandono"]
    return train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])


def entrenar(X_train, y_train, config):
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
        random_state=config["semillas"]["modelo"],
    )
    modelo.fit(X_train, y_train)
    return modelo


def pipeline_completo(config):
    datos = generar_datos(config["n_filas"], config["semillas"]["datos"])
    X_train, X_test, y_train, y_test = dividir(datos, config)
    modelo = entrenar(X_train, y_train, config)
    exactitud = modelo.score(X_test, y_test)
    predicciones = modelo.predict(X_test)
    return modelo, exactitud, predicciones
'''

with open(os.path.join(WORK, "src", "pipeline.py"), "w") as f:
    f.write(codigo_pipeline)

spec = importlib.util.spec_from_file_location("pipeline", os.path.join(WORK, "src", "pipeline.py"))
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
print("src/pipeline.py escrito e importado.")

In [ ]:
# Ejecución 1 y ejecución 2 de la misma pipeline, con la misma config: deben coincidir
_, exactitud_1, predicciones_1 = pipeline.pipeline_completo(config)
_, exactitud_2, predicciones_2 = pipeline.pipeline_completo(config)

print("pipeline_completo — exactitud ejecución 1:", exactitud_1)
print("pipeline_completo — exactitud ejecución 2:", exactitud_2)
print("¿misma exactitud en las dos ejecuciones?", exactitud_1 == exactitud_2)
print("¿mismas predicciones en las dos ejecuciones?", list(predicciones_1) == list(predicciones_2))
assert exactitud_1 == exactitud_2
assert list(predicciones_1) == list(predicciones_2)

**¿Qué observas?** Con las tres semillas de `config.yaml` fijadas (datos, división y modelo), `pipeline_completo` da exactamente la misma exactitud y las mismas predicciones en las dos ejecuciones. Ahora comparamos con una versión que solo olvida fijar la semilla del modelo — una sola llamada, entre varias, sin `random_state` — para ver si sigue siendo reproducible.

In [ ]:
def entrenar_sin_semilla(X_train, y_train, config):
    """Igual que entrenar(), pero "olvida" random_state en el modelo."""
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
    )
    modelo.fit(X_train, y_train)
    return modelo


# Datos y división SÍ tienen semilla (2 de las 3 fuentes de la sección 3 están controladas)
datos_config = pipeline.generar_datos(config["n_filas"], config["semillas"]["datos"])
X_train, X_test, y_train, y_test = pipeline.dividir(datos_config, config)

# Simulamos "máquina A" y "máquina B": cada una hereda un estado global de NumPy distinto,
# algo real y fuera de tu control, que aquí fijamos solo para que esta demostración sea reproducible en el manual
np.random.seed(1001)
modelo_a = entrenar_sin_semilla(X_train, y_train, config)
exactitud_a = modelo_a.score(X_test, y_test)

np.random.seed(2002)
modelo_b = entrenar_sin_semilla(X_train, y_train, config)
exactitud_b = modelo_b.score(X_test, y_test)

print("sin semilla en el modelo — exactitud «máquina A»:", exactitud_a)
print("sin semilla en el modelo — exactitud «máquina B»:", exactitud_b)
print("¿misma exactitud en las dos «máquinas»?", exactitud_a == exactitud_b)
assert exactitud_a != exactitud_b

**¿Qué observas?** Con los mismos datos y la misma división (dos de las tres fuentes de aleatoriedad controladas), olvidar `random_state` en una sola llamada — la del modelo — basta para que "máquina A" y "máquina B" entrenen modelos distintos y obtengan exactitudes distintas. Es la sección 3.1 del capítulo hecha código: **un `random_state` que falta en una sola llamada, en un proyecto con varios pasos**, es el caso más traicionero porque el resto del *pipeline* sí parece controlado.

---
### 6.1 Una prueba automática, con `pytest`

Por último, escribimos `tests/test_pipeline.py`: una prueba que falla si `pipeline_completo` deja de ser reproducible. La ejecutamos con `pytest` de verdad, en un subproceso, tal como se ejecutaría en la clase 6 o en un flujo de integración continua (clase 9).

In [ ]:
codigo_test = '''
import os
import sys
import yaml

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from src.pipeline import pipeline_completo


def test_pipeline_es_reproducible():
    ruta_config = os.path.join(os.path.dirname(__file__), "..", "config", "config.yaml")
    with open(ruta_config) as f:
        config = yaml.safe_load(f)
    _, exactitud_1, predicciones_1 = pipeline_completo(config)
    _, exactitud_2, predicciones_2 = pipeline_completo(config)
    assert exactitud_1 == exactitud_2
    assert list(predicciones_1) == list(predicciones_2)
'''

with open(os.path.join(WORK, "tests", "test_pipeline.py"), "w") as f:
    f.write(codigo_test)

resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=WORK, capture_output=True, text=True)
coincidencias = re.findall(r"(\d+) passed", resultado.stdout)
n_pasadas = coincidencias[0] if coincidencias else "0"
print("tests/test_pipeline.py ejecutado con pytest. Pruebas superadas:", n_pasadas)
assert resultado.returncode == 0
assert n_pasadas == "1"

**¿Qué observas?** `pytest` confirma, de forma automática, lo que ya vimos a mano: la prueba pasa (`1 passed`) porque `pipeline_completo`, con las tres semillas fijadas en `config.yaml`, es reproducible. A partir de ahora, si alguien introduce sin querer un `RandomForestClassifier()` sin `random_state` dentro de `src/pipeline.py`, esta misma prueba fallaría y lo avisaría, sin depender de que alguien lo note a simple vista.

---
## Resumen de la clase

- **"En mi máquina funciona"** es un síntoma: si el mismo código con los mismos datos da resultados distintos, el problema está en el código, no en quién lo ejecuta.
- **Dependencias bloqueadas** (`requirements.txt` o `pyproject.toml`, con versiones exactas) son necesarias pero no suficientes: fijan paquetes, no el entorno completo (eso es Docker, clase 8).
- Un proyecto pequeño tiene **al menos tres fuentes de aleatoriedad** (NumPy, división train/test, el modelo); basta con olvidar una — a menudo el *shuffle* interno o la inicialización de pesos del modelo — para perder la reproducibilidad completa.
- Una **estructura de carpetas** con responsabilidades claras (`src/`, `config/`, `tests/`), inspirada en Cookiecutter Data Science, y una **configuración separada del código** (`config.yaml`) hacen que cambiar un parámetro no obligue a tocar código.
- `proyecto_churn/` demuestra, con `pytest` de por medio, que la misma *pipeline* con las tres semillas fijadas da el mismo resultado en dos ejecuciones, mientras que olvidar solo una (la del modelo) ya lo rompe.

*Continúa con la clase 3: Versionado de datos y modelos.*